# 05_01 Loss and epochs

*📖 Book: sections 5.1.2 (loss) and 5.2 (training)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** Training prints a loss every so often. We need to read it correctly: print totals where you meant averages, or print too rarely, and big jumps in the loss stay hidden.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a loss number means, for a good guess and a bad one
- Why we report the average loss, not the total
- How to read a loss curve, and why a few snapshots can hide spikes
- Why the same seed gives the same results every run

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **loss curve** | the loss after each epoch, one after another; it shows how training went | `3.88 → … → 0.71` |
| **seed** | a starting number for randomness; the same seed gives the same "random" numbers | `torch.manual_seed(42)` |

Words you've met before: **weight** ([01_00](../01_foundations/01_00_chapter_overview.ipynb)), **loss** ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)), **training step** ([05_00](../05_training_on_the_odyssey/05_00_chapter_overview.ipynb)), **optimizer** ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)), **epoch** ([05_00](../05_training_on_the_odyssey/05_00_chapter_overview.ipynb)), **chunk** ([02_04](../02_text_to_numbers/02_04_sliding_window.ipynb)), **score** ([01_04](../01_foundations/01_04_our_model_as_a_black_box.ipynb)).

## 3. The idea

### ❓ What's the loss, exactly?

A **mistake score**. For each position, the model gives every possible next token a score, and the highest score is its guess. The loss is small when the model gave the right answer a high score, and large when it didn't. Lower is better.

### ❓ What number should I hope for?

Random guessing among our 137 tokens gives a loss of about **4.9**. Our model, trained on Book 1, reaches an average of about 2.38. So it has clearly learned something, but it's far from perfect.

### ❓ What does a loss of 2.38 mean?

Here's a rough picture: e^2.38 ≈ 10.8, so the model is about as unsure as if it were choosing between **11** characters. At the start it was choosing between all 137. (`e` is a special number in maths, about 2.718. The loss is measured on that scale.)

### ❓ Why are the targets the inputs shifted by one?

Because the job is always the same: guess the next character. If the input is `ἄνδρ`, the targets are `νδρα`. After `ἄ` the answer is `ν`, after `ἄν` it's `δ`, and so on. So one chunk of 24 characters gives the model **24 practice questions** at once.

### ❓ What's an epoch?

One full pass through all the training text, chunk by chunk. Training repeats many epochs, so the model sees the same text again and again.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ What's the loss for one real flashcard?

Take the flashcard `ἄνδρ → α` from [02_04](../02_text_to_numbers/02_04_sliding_window.ipynb). The model gives each of the 137 possible next characters a chance. The loss only looks at the chance it gave the **right** answer, `α`:

```
flashcard: ἄνδρ → α
untrained: chance for α = 0.0056, loss = 5.180
trained:   chance for α = 0.1194, loss = 2.125
```

👉 **Predict first:** which model gets the lower loss?

In [ ]:
import torch
import torch.nn.functional as F

from homer_ai.lessons import load_homer_model
from homer_ai.model import HomerGreekLLM

tok, trained = load_homer_model()  # read-only: the saved model, if there is one (05_06)
torch.manual_seed(42)
models = [("untrained", HomerGreekLLM(tok.vocab_size, 32))]
if trained is not None:
    models.append(("trained", trained))

inputs = torch.tensor(tok.encode("ἄνδρ"), dtype=torch.long)
target = torch.tensor(tok.encode("α"), dtype=torch.long)

print("flashcard: ἄνδρ → α")
for name, model in models:
    with torch.no_grad():
        scores = model(inputs)[-1:]  # the scores after the last character
    chance = F.softmax(scores, dim=-1)[0, target].item()
    loss = F.cross_entropy(scores, target).item()
    print(f"{name + ':':10} chance for α = {chance:.4f}, loss = {loss:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The untrained model gave `α` a tiny chance (about 1 in 180), so its loss is high. The trained one gave it about 0.12, so its loss is lower. `[-1:]` keeps only the last row of scores, which is the guess after `ἄνδρ`.

</details>

### ❓ How does the chance turn into the loss?

The bigger the chance for the right answer, the smaller the loss. Some real pairs:

```
chance 0.90  →  loss 0.105
chance 0.50  →  loss 0.693
chance 0.12  →  loss 2.120
chance 0.01  →  loss 4.605
```

A chance of 1 (completely sure, and right) would give a loss of 0. In maths this is the **negative logarithm** of the chance. You don't need the maths here, only the pattern.

👉 **Predict first:** is a chance of 0.5 closer to loss 0 or to loss 5?

In [ ]:
import math

for chance in [0.90, 0.50, 0.12, 0.01]:
    print(f"chance {chance:.2f}  →  loss {-math.log(chance):.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The loss grows slowly at first, then fast as the chance gets small. So being confidently wrong is what costs the most.

</details>

### ❓ Can I see how the chance turns into the loss?

Draw the loss for every chance from 0 to 1, and mark the four real pairs from above.

👉 **Predict first:** where will the curve be steepest: near chance 0, or near chance 1?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

chances = np.linspace(0.005, 1, 200)
fig = studio_figure("From chance to loss", "The less sure it is, the more it costs")
ax = studio_axes(fig)
ax.plot(chances, -np.log(chances), color=MUTED, linewidth=3)
for chance in [0.90, 0.50, 0.12, 0.01]:
    loss = -math.log(chance)
    ax.plot(chance, loss, "o", color=ACCENT, markersize=10)
    ax.text(
        chance + 0.025, loss + 0.1, f"{chance:.2f} → {loss:.2f}",
        color=INK, family="monospace", fontsize=11,
    )
ax.set_xlabel("chance the model gave the right answer")
ax.set_ylabel("loss")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The curve is flat and low on the right (being sure and right costs little). On the left it climbs steeply, and that's where the model gave the right answer almost no chance.

</details>

### ❓ What does the loss look like for a good guess and a bad guess?

`F.cross_entropy` is PyTorch's function for the loss. Here there are 3 possible tokens, and the right answer is token 0. `torch.tensor(...)` makes a **tensor**, PyTorch's list of numbers.

👉 **Predict first:** which guess gets the lower loss?

In [ ]:
import torch
import torch.nn.functional as F

right_answer = torch.tensor([0])
good_guess = torch.tensor([[3.0, 0.0, 0.0]])  # high score for token 0
bad_guess = torch.tensor([[0.0, 0.0, 3.0]])  # high score for token 2

print("good guess loss:", F.cross_entropy(good_guess, right_answer).item())
print("bad guess loss:", F.cross_entropy(bad_guess, right_answer).item())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 0.09 for the good guess and about 3.09 for the bad one. Lower loss, better guess.

</details>

### ❓ Should we add up the losses of all chunks, or average them?

Average them. A total grows with the amount of text, so a longer book would look "worse" even if every guess were just as good.

👉 **Predict first:** what are the total and the average of these 4 chunk losses?

In [ ]:
chunk_losses = [2.0, 1.5, 1.8, 1.7]
print("total:", sum(chunk_losses))
print("average:", sum(chunk_losses) / len(chunk_losses))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Total 7.0, average 1.75. With 10× the chunks the total would be 10× bigger, but the average would stay at 1.75. That's why our training code prints the **average**.

</details>

### ❓ What does training look like on a small text?

This cell sets up a tiny training run on the first 5 lines of Book 1. It finishes in a second or two, and saves nothing. It does the four training steps from the overview for every chunk, and records the average loss of each epoch. Read `tiny_train` once, slowly: the next notebooks use this same function (a copy is kept in `src/homer_ai/lessons.py`, so they can import it).

Just run it. It only sets things up, so nothing is printed.

In [ ]:
import torch
import torch.nn.functional as F

from homer_ai.model import HomerGreekLLM
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

# A short practice text: the first 5 lines of Book 1
practice_text = "\n".join(load_books(BOOKS_DIR)[0].splitlines()[:5])
tokenizer = CharTokenizer(practice_text)
data = torch.tensor(tokenizer.encode(practice_text))


def tiny_train(seed, epochs, lr=0.01, chunk_size=24):
    torch.manual_seed(seed)
    model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    average_losses = []
    for _epoch in range(epochs):
        losses = []
        for i in range(0, len(data) - chunk_size, chunk_size):
            inputs = data[i : i + chunk_size]
            targets = data[i + 1 : i + chunk_size + 1]
            loss = F.cross_entropy(model(inputs), targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        average_losses.append(sum(losses) / len(losses))
    return average_losses

### ❓ Your turn: can you write the three training lines?

Inside `tiny_train`, three lines do the learning: `zero_grad`, `backward`, `step`. Here's one model and one chunk of 24 characters, with the loss already measured. You write the three lines, then measure the loss on the same chunk again.

👉 **Predict first:** after one single step, will the loss on this chunk be lower? By a lot, or by a little?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
torch.manual_seed(0)
step_model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
step_optimizer = torch.optim.AdamW(step_model.parameters(), lr=0.01)
step_inputs, step_targets = data[0:24], data[1:25]

loss_before = F.cross_entropy(step_model(step_inputs), step_targets)
print("loss before:", round(loss_before.item(), 3))

# 👉 do: write the three training lines here, with step_optimizer and loss_before
# 💡 hint: forget the old gradients, work out the new ones, take one step

# 👉 do: measure the loss on the same chunk again, the way loss_before was measured
loss_after = None

In [ ]:
from homer_ai.lessons import check

check(
    "the loss after one training step",
    loss_after,
    test=lambda after: after < loss_before,
    expected=f"a loss below {loss_before.item():.3f}",
    coach="The same loss as before means the weights didn't change. Is one of "
    "the three lines missing? backward() only works out the gradients. It's "
    "step() that changes the weights.",
)

<details><summary>💡 Hint</summary>

Look inside `tiny_train`, right after the line that computes `loss`. The same three lines, with `step_optimizer` as the optimizer and `loss_before` as the loss.

</details>

<details><summary>✅ Answer</summary>

```python
torch.manual_seed(0)
step_model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
step_optimizer = torch.optim.AdamW(step_model.parameters(), lr=0.01)
step_inputs, step_targets = data[0:24], data[1:25]

loss_before = F.cross_entropy(step_model(step_inputs), step_targets)
print("loss before:", round(loss_before.item(), 3))

step_optimizer.zero_grad()
loss_before.backward()
step_optimizer.step()

loss_after = F.cross_entropy(step_model(step_inputs), step_targets)
print("loss after: ", round(loss_after.item(), 3))
```

The loss drops from about 3.99 to about 3.81. One step is one small nudge to every weight, like the single nudge of `w` in [01_03](../01_foundations/01_03_how_a_program_learns.ipynb). Training is this step, thousands of times.

</details>

### ❓ Does the loss go down?

👉 **Predict first:** after 60 epochs, will the last loss be lower than the first?

In [ ]:
losses = tiny_train(seed=42, epochs=60)
print("first epoch:", round(losses[0], 3))
print("last epoch:", round(losses[-1], 3))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About 3.88 at the start and 0.71 at the end. So the model is learning the 5 lines.

But why 3.88 and not 4.9? The 5-line text has only **52** different characters, not 137, and random guessing among 52 scores about 3.95. And the first number is already the average over the whole first epoch, so the model has learned a little by then.

</details>

### ❓ Is it enough to look at a few snapshots?

No. Printing only every so often is like judging a whole football match from 4 photos. Let's keep only every 15th loss and compare that with the full list.

👉 **Predict first:** will the full list look as smooth as the snapshots?

In [ ]:
print("every 15th epoch:", [round(x, 2) for x in losses[14::15]])
print()
print("every epoch:", [round(x, 2) for x in losses])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The snapshots go smoothly down, but the full list wobbles up and down along the way.

On a longer run the snapshots hide even more. In [05_02](05_02_learning_rate.ipynb) you'll train for 400 epochs. Printed every 100 epochs, that run reads `0.46 → 0.15 → 0.21 → 0.48`: four calm-looking numbers. Count every epoch, and the loss jumps back up 14 times in between.

</details>

### ❓ Why do I get exactly the same numbers every time I run it?

The model starts with random weights. The **seed** fixes the starting point of that randomness, so the "random" numbers are the same every run.

👉 **Predict first:** two runs with seed 42, and one with seed 7. Which ones match?

In [ ]:
run_a = tiny_train(seed=42, epochs=20)
run_b = tiny_train(seed=42, epochs=20)
run_c = tiny_train(seed=7, epochs=20)

print("seed 42 twice, same?", run_a == run_b)
print("seed 42 vs seed 7, same?", run_a == run_c)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same seed gives exactly the same numbers, and a different seed doesn't. That's handy when you change the code: same seed, same numbers, nothing broken.

</details>

## 5. Full example: a loss curve

A 100-epoch tiny run, printed every 10 epochs.

👉 **Predict first:** will every printed loss be lower than the one before?

In [ ]:
curve = tiny_train(seed=42, epochs=100)
for epoch in range(9, 100, 10):
    print(f"epoch {epoch + 1:3} | average loss {curve[epoch]:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The trend goes down (about 1.60 → 0.46), but not at every step. Epoch 60 and epoch 90 are a little higher than the epoch before.

</details>

### ❓ Can I see the whole curve?

The numbers above show every 10th epoch. Here's every epoch, with the printed ones marked.

👉 **Predict first:** will the line fall smoothly, or wobble?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, MUTED, studio_axes, studio_figure

fig = studio_figure(
    "A tiny training run, all 100 epochs",
    f"The loss falls from {curve[0]:.2f} to {curve[-1]:.2f}, with wobbles",
)
ax = studio_axes(fig)
ax.plot(range(1, 101), curve, color=MUTED, linewidth=2)
printed = list(range(10, 101, 10))
ax.plot(printed, [curve[e - 1] for e in printed], "o", color=ACCENT, markersize=8)
ax.set_xlabel("epoch")
ax.set_ylabel("average loss")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The grey line wobbles on its way down, and the orange dots (the epochs we printed) hide those wobbles. Same warning as before: look at every epoch, not just a few.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ The loss suddenly jumped up during training. Is something broken?

Not necessarily. Usually the steps are too big, and the model overshoots. That's the next notebook, [05_02](05_02_learning_rate.ipynb).

### ❓ A log says the loss is 1966. Is that terrible?

It depends what was printed. As a **total** over all the chunks of Book 1, 1966 is fine: it's an average of about 2.40 per chunk. So always check whether a number is a total or an average.

### ❓ Why print the loss at all?

It's the only way to see whether training is working. A curve that goes down means the model is learning.

## 6. Recap

- The loss is a mistake score, and lower is better (random guessing on our 137 tokens: about 4.9).
- Report the average loss, and look at it often. Rare snapshots hide spikes.
- The same seed gives the same numbers, so runs can be compared.

**You can now:**

- turn a chance into a loss, and say what loss random guessing gives
- write the three lines of a training step: `zero_grad`, `backward`, `step`
- read a loss curve, epoch by epoch

**Next:** [05_02: Learning rate](05_02_learning_rate.ipynb): why does the loss jump up?